# Análisis Exploratorio — Machine Predictive Maintenance

**Fuente**: Kaggle — *Machine Predictive Maintenance Classification Dataset*  
**Ruta**: `data/predictive_maintenance.csv`

El objetivo es identificar qué variables tienen poder discriminante, qué variables derivadas conviene construir, y qué restricciones impone la distribución de clases al modelado.

## 1. Descripción del dataset

10,000 observaciones de un proceso industrial genérico, sintético. Cada fila es un punto de operación independiente — no es una serie de tiempo.

| Columna | Tipo | Descripción |
|---|---|---|
| `UDI` | Entero | Identificador único |
| `Product ID` | Cadena | ID del producto (prefijo = tipo) |
| `Type` | Categórica | Tipo de máquina: H (alto), M (medio), L (bajo) |
| `Air temperature [K]` | Continua | Temperatura del aire ambiente |
| `Process temperature [K]` | Continua | Temperatura del proceso |
| `Rotational speed [rpm]` | Continua | Velocidad de rotación |
| `Torque [Nm]` | Continua | Par mecánico |
| `Tool wear [min]` | Continua | Tiempo acumulado de uso de herramienta |
| `Target` | Binaria | 0 = sin fallo, 1 = fallo |
| `Failure Type` | Categórica | Causa específica del fallo |

**Tipos de fallo**: Tool Wear Failure, Heat Dissipation Failure, Power Failure, Overstrain Failure, Random Failures.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

TEMPLATE = 'plotly_white'
DATA_DIR = Path('data')

## 2. Carga e inspección inicial

In [ ]:
# encoding='utf-8-sig' para manejar el BOM del archivo
df = pd.read_csv(DATA_DIR / 'predictive_maintenance.csv', encoding='utf-8-sig')

df = df.rename(columns={
    'UDI': 'id',
    'Product ID': 'product_id',
    'Type': 'machine_type',
    'Air temperature [K]': 'air_temp',
    'Process temperature [K]': 'process_temp',
    'Rotational speed [rpm]': 'speed',
    'Torque [Nm]': 'torque',
    'Tool wear [min]': 'tool_wear',
    'Target': 'target',
    'Failure Type': 'failure_type'
})

print(f"Dimensiones: {df.shape}")
print(f"\nTipos de datos:")
print(df.dtypes)
print(f"\nValores nulos: {df.isnull().sum().sum()}")
df.head()

In [ ]:
CONTINUOUS = ['air_temp', 'process_temp', 'speed', 'torque', 'tool_wear']
df[CONTINUOUS].describe().round(2)

## 3. Distribución del target — desbalance de clases

Con ~3.6% de fallos, un clasificador que siempre prediga "sin fallo" alcanza 96.4% de exactitud sin aprender nada. Las métricas relevantes son **Precision, Recall y AUC-PR**, no la exactitud simple.

El dataset tiene dos columnas relacionadas con el fallo: `target` (binaria) y `failure_type` (categórica). La columna `failure_type` es la fuente de verdad — contiene la causa específica de cada evento. `target` debería ser su proyección binaria: 1 si `failure_type != 'No Failure'`, 0 en caso contrario. En la sección de código se verifica esta relación y se construye el target limpio a partir de `failure_type`.

In [ ]:
# Distribución del target original
counts = df['target'].value_counts().reset_index()
counts.columns = ['target', 'n']
counts['label'] = counts['target'].map({0: 'No failure', 1: 'Failure'})
counts['pct'] = (counts['n'] / len(df) * 100).round(2)

fig = px.bar(counts, x='label', y='n', color='label',
             text=counts['pct'].apply(lambda x: f'{x}%'),
             title='Distribución del target binario',
             labels={'n': 'Observaciones', 'label': ''},
             color_discrete_map={'No failure': 'steelblue', 'Failure': 'tomato'},
             template=TEMPLATE)
fig.update_traces(textposition='outside')
fig.show()

# Tipos de fallo
counts_by_type = df['failure_type'].value_counts().reset_index()
counts_by_type.columns = ['failure_type', 'n']

fig2 = px.bar(counts_by_type, x='failure_type', y='n',
              title='Distribución por tipo de fallo',
              labels={'n': 'Observaciones', 'failure_type': 'Tipo de fallo'},
              template=TEMPLATE)
fig2.show()

# Construir target limpio desde failure_type
df['failure'] = (df['failure_type'] != 'No Failure').astype(int)
discrepancias = (df['target'] != df['failure']).sum()
print(f"Discrepancias entre target original y failure_type binarizada: {discrepancias}")
print(f"Tasa de fallos (failure_type binarizada): {df['failure'].mean()*100:.2f}%")

## 4. Tipo de máquina

`machine_type` (H/M/L) categoriza la especificación del equipo. En el proceso de síntesis del dataset, los umbrales de fallo por desgaste de herramienta varían por tipo: L ≤ 200 min, M ≤ 150 min, H ≤ 200 min.

In [ ]:
table = df.groupby(['machine_type', 'failure']).size().reset_index(name='n')
table['status'] = table['failure'].map({0: 'No failure', 1: 'Failure'})

fig = px.bar(table, x='machine_type', y='n', color='status', barmode='group',
             title='Fallos por tipo de máquina',
             labels={'n': 'Observaciones', 'machine_type': 'Tipo'},
             color_discrete_map={'No failure': 'steelblue', 'Failure': 'tomato'},
             template=TEMPLATE)
fig.show()

rate_by_type = df.groupby('machine_type').agg(
    n=('failure', 'count'),
    failures=('failure', 'sum')
).assign(rate_pct=lambda x: (x['failures'] / x['n'] * 100).round(2))
print(rate_by_type)

## 5. Distribución de variables continuas

Las temperaturas tienen rangos estrechos (~10 K de variación); la diferencia entre ellas puede ser más informativa que cada una por separado. `speed` y `torque` están relacionadas por la potencia mecánica P = τ·ω.

In [ ]:
colors = {0: 'steelblue', 1: 'tomato'}
labels = {0: 'No failure', 1: 'Failure'}

fig = make_subplots(rows=2, cols=3, subplot_titles=CONTINUOUS)

for i, col in enumerate(CONTINUOUS):
    row = i // 3 + 1
    col_idx = i % 3 + 1
    for val in [0, 1]:
        data = df[df['failure'] == val][col]
        fig.add_trace(
            go.Histogram(
                x=data, name=labels[val],
                marker_color=colors[val],
                opacity=0.6, showlegend=(i == 0)
            ),
            row=row, col=col_idx
        )

fig.update_layout(
    barmode='overlay', template=TEMPLATE,
    title='Distribución de variables continuas — sin fallo vs fallo',
    height=520
)
fig.show()

In [ ]:
fig = make_subplots(rows=2, cols=3, subplot_titles=CONTINUOUS)

for i, col in enumerate(CONTINUOUS):
    row = i // 3 + 1
    col_idx = i % 3 + 1
    for val in [0, 1]:
        data = df[df['failure'] == val][col]
        fig.add_trace(
            go.Box(
                y=data, name=labels[val],
                marker_color=colors[val],
                showlegend=(i == 0)
            ),
            row=row, col=col_idx
        )

fig.update_layout(
    template=TEMPLATE,
    title='Box plots por estado de fallo',
    height=520
)
fig.show()

## 6. Correlaciones entre variables continuas

En un sistema rotativo bajo carga nominal, la potencia P = τ·ω se mantiene aproximadamente constante, lo que genera correlación negativa entre `torque` y `speed`. Desde el punto de vista de features, no son independientes: representan el mismo punto de operación.

In [ ]:
corr = df[CONTINUOUS].corr()

fig = go.Figure(go.Heatmap(
    z=corr.values,
    x=corr.columns.tolist(),
    y=corr.index.tolist(),
    colorscale='RdBu_r',
    zmid=0, zmin=-1, zmax=1,
    text=[[f'{v:.2f}' for v in row] for row in corr.values],
    texttemplate='%{text}'
))
fig.update_layout(
    title='Matriz de correlación — variables continuas',
    template=TEMPLATE
)
fig.show()

high_corr_pairs = [
    (i, j, round(corr.loc[i, j], 3))
    for idx_i, i in enumerate(corr.index)
    for idx_j, j in enumerate(corr.columns)
    if idx_i < idx_j and abs(corr.loc[i, j]) > 0.5
]
print("Pares con |r| > 0.5:")
for a, b, r in high_corr_pairs:
    print(f"  {a} — {b}: r = {r}")

## 7. Variables derivadas

Las causas de fallo documentadas en el dataset tienen una base física directa:

| Variable | Fórmula | Mecanismo de fallo |
|---|---|---|
| `power` | τ × (ω × 2π/60) [W] | Power Failure: potencia fuera de rango |
| `temp_diff` | process_temp − air_temp [K] | Heat Dissipation Failure: diferencial insuficiente |
| `strain` | torque × tool_wear | Overstrain Failure: carga alta con herramienta desgastada |

In [ ]:
df['power'] = df['torque'] * (df['speed'] * 2 * np.pi / 60)
df['temp_diff'] = df['process_temp'] - df['air_temp']
df['strain'] = df['torque'] * df['tool_wear']

DERIVED = ['power', 'temp_diff', 'strain']

fig = make_subplots(rows=1, cols=3, subplot_titles=DERIVED)

for i, col in enumerate(DERIVED):
    for val in [0, 1]:
        data = df[df['failure'] == val][col]
        fig.add_trace(
            go.Histogram(
                x=data, name=labels[val],
                marker_color=colors[val],
                opacity=0.6, showlegend=(i == 0)
            ),
            row=1, col=i + 1
        )

fig.update_layout(
    barmode='overlay', template=TEMPLATE,
    title='Variables derivadas — sin fallo vs fallo',
    height=400
)
fig.show()

## 8. Relaciones bivariadas

Se busca si los fallos forman regiones separables en el espacio de pares de variables. Separabilidad lineal sugiere que un modelo simple es suficiente; fallos dispersos o en bordes de región indican que se necesita capturar interacciones.

In [ ]:
fig = px.scatter(
    df, x='speed', y='torque', color='failure_type',
    opacity=0.5,
    title='Speed vs Torque — tipos de fallo',
    labels={'speed': 'Speed [rpm]', 'torque': 'Torque [Nm]'},
    template=TEMPLATE
)
fig.show()

In [ ]:
fig = px.scatter(
    df, x='tool_wear', y='torque', color='failure_type',
    opacity=0.5,
    title='Tool Wear vs Torque — tipos de fallo',
    labels={'tool_wear': 'Tool wear [min]', 'torque': 'Torque [Nm]'},
    template=TEMPLATE
)
fig.show()

fig2 = px.scatter(
    df, x='speed', y='temp_diff', color='failure_type',
    opacity=0.5,
    title='Speed vs ΔTemp — tipos de fallo',
    labels={'speed': 'Speed [rpm]', 'temp_diff': 'ΔTemp [K]'},
    template=TEMPLATE
)
fig2.show()

## 9. Análisis por tipo de fallo

Cada mecanismo de fallo deja una firma distinta en las variables. Verificar que las variables derivadas (power, temp_diff, strain) efectivamente separan los tipos correspondientes es una validación de la hipótesis física antes de usarlas en el modelo.

In [ ]:
failures_df = df[df['failure_type'] != 'No Failure'].copy()

ALL_FEATURES = CONTINUOUS + DERIVED
failure_types = sorted(failures_df['failure_type'].unique())
failure_colors = dict(zip(failure_types, px.colors.qualitative.Set2[:len(failure_types)]))

fig = make_subplots(rows=3, cols=3, subplot_titles=ALL_FEATURES)

for i, col in enumerate(ALL_FEATURES):
    row = i // 3 + 1
    col_idx = i % 3 + 1
    for ft in failure_types:
        data = failures_df[failures_df['failure_type'] == ft][col]
        fig.add_trace(
            go.Box(
                y=data, name=ft,
                marker_color=failure_colors[ft],
                showlegend=(i == 0)
            ),
            row=row, col=col_idx
        )

fig.update_layout(
    template=TEMPLATE,
    title='Distribución por tipo de fallo — variables originales y derivadas',
    height=800
)
fig.show()

## 10. Desgaste de herramienta

`Tool Wear Failure` ocurre cuando el desgaste acumulado supera un umbral que depende del tipo de máquina. Se verifica si la tasa de fallo crece con el decil de desgaste, lo que confirmaría que `tool_wear` es un predictor monotónico para este tipo específico.

In [ ]:
fig = go.Figure()
for val in [0, 1]:
    fig.add_trace(go.Histogram(
        x=df[df['failure'] == val]['tool_wear'],
        name=labels[val],
        marker_color=colors[val],
        opacity=0.6
    ))
fig.update_layout(
    barmode='overlay', template=TEMPLATE,
    title='Distribución del desgaste por estado de fallo',
    xaxis_title='Tool wear [min]', yaxis_title='Frecuencia'
)
fig.show()

df['wear_decile'] = pd.qcut(df['tool_wear'], q=10, labels=False)
rate_by_decile = df.groupby('wear_decile').agg(
    n=('failure', 'count'),
    failures=('failure', 'sum'),
    max_wear=('tool_wear', 'max')
).assign(rate=lambda x: x['failures'] / x['n']).reset_index()

fig2 = px.bar(
    rate_by_decile, x='wear_decile', y='rate',
    hover_data=['max_wear', 'n', 'failures'],
    title='Tasa de fallo por decil de desgaste',
    labels={'rate': 'Tasa de fallo', 'wear_decile': 'Decil (0 = menor desgaste)'},
    template=TEMPLATE
)
fig2.show()

## 11. Poder discriminante univariado

AUC calculado por variable individualmente. AUC = 0.5 equivale a azar; AUC ≥ 0.7 indica separabilidad útil. Permite comparar directamente las variables originales contra las derivadas.

In [ ]:
from sklearn.metrics import roc_auc_score

df['machine_type_enc'] = df['machine_type'].map({'H': 2, 'M': 1, 'L': 0})

CANDIDATES = CONTINUOUS + DERIVED + ['machine_type_enc']

auc_scores = {}
for col in CANDIDATES:
    auc = roc_auc_score(df['failure'], df[col])
    auc_scores[col] = round(max(auc, 1 - auc), 4)

auc_df = (
    pd.DataFrame.from_dict(auc_scores, orient='index', columns=['AUC'])
    .sort_values('AUC', ascending=False)
    .reset_index()
    .rename(columns={'index': 'variable'})
)

fig = px.bar(
    auc_df, x='variable', y='AUC',
    title='AUC univariado por variable',
    labels={'variable': 'Variable', 'AUC': 'AUC'},
    template=TEMPLATE
)
fig.add_hline(y=0.5, line_dash='dash', line_color='gray', annotation_text='azar')
fig.add_hline(y=0.7, line_dash='dot', line_color='orange', annotation_text='umbral útil')
fig.show()

print(auc_df.to_string(index=False))

## 12. Síntesis

### Hallazgos

| Aspecto | Observación |
|---|---|
| Desbalance | ~3.6% de fallos. Requiere class_weight, oversampling (SMOTE), o ajuste del umbral de decisión |
| Variables originales | `torque` y `speed` son las más discriminantes; temperaturas individuales, poco informativas |
| Variables derivadas | `power` = τ·ω, `temp_diff`, `strain` = torque × tool_wear capturan directamente los mecanismos de fallo |
| Correlación | `torque` y `speed` negativamente correlacionados (P ≈ constante) — parcialmente redundantes como features separadas |
| Target | Construir desde `failure_type != 'No Failure'`; la columna `target` original tiene 9 discrepancias |

### Secuencia de modelado

1. **Clasificación binaria** — baseline con regresión logística. Features: variables continuas + derivadas + `machine_type_enc`. Métrica: AUC-ROC y F1 de la clase positiva
2. **Random Forest / XGBoost** — captura interacciones torque × tool_wear sin construirlas explícitamente. Evaluar importancia de features para confirmar o descartar las derivadas
3. **Clasificación multiclase (failure_type)** — extender una vez que el modelo binario sea estable. Las clases son pequeñas (18–112 ejemplos); considerar macro-F1 y one-vs-rest
4. **Calibración** — con desbalance severo las probabilidades de salida tienden a estar mal calibradas. Aplicar Platt scaling o isotonic regression antes de fijar umbrales de decisión

### Features de entrada

```
air_temp, process_temp, speed, torque, tool_wear,
power, temp_diff, strain, machine_type_enc
```

> Los datos son sintéticos: las fronteras de decisión son más limpias que en datos industriales reales, donde intervienen drift de sensores y efectos no modelados.